# Glinear2

https://github.com/fastino-ai/GLiNER2

-> 한국어 잘 안되네... ㅠㅠ

In [8]:
from gliner2 import GLiNER2

# Load model once, use everywhere
# extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
extractor = GLiNER2.from_pretrained("fastino/gliner2-large-v1")

# Extract entities in one line
text = "Apple CEO Tim Cook announced iPhone 15 in Cupertino yesterday."
result = extractor.extract_entities(text, ["company", "person", "product", "location"])

print(result)
# {'entities': {'company': ['Apple'], 'person': ['Tim Cook'], 'product': ['iPhone 15'], 'location': ['Cupertino']}}

🧠  Model Configuration
Encoder model      : microsoft/deberta-v3-large
Counting layer     : count_lstm
Token pooling      : first


You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'entities': {'company': ['Apple'], 'person': ['Tim Cook'], 'product': ['iPhone 15'], 'location': ['Cupertino']}}


In [3]:
# Basic entity extraction
entities = extractor.extract_entities(
    "Patient received 400mg ibuprofen for severe headache at 2 PM.", ["medication", "dosage", "symptom", "time"]
)
print(entities)
# Output: {'entities': {'medication': ['ibuprofen'], 'dosage': ['400mg'], 'symptom': ['severe headache'], 'time': ['2 PM']}}

# Enhanced with descriptions for medical accuracy
entities = extractor.extract_entities(
    "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    {
        "medication": "Names of drugs, medications, or pharmaceutical substances",
        "dosage": "Specific amounts like '400mg', '2 tablets', or '5ml'",
        "symptom": "Medical symptoms, conditions, or patient complaints",
        "time": "Time references like '2 PM', 'morning', or 'after lunch'",
    },
)
print(entities)
# Same output but with higher accuracy due to context descriptions

{'entities': {'medication': ['ibuprofen'], 'dosage': ['400mg'], 'symptom': ['headache'], 'time': ['2 PM']}}
{'entities': {'medication': ['ibuprofen'], 'dosage': ['400mg'], 'symptom': ['headache'], 'time': ['2 PM']}}


In [4]:
# Sentiment analysis
result = extractor.classify_text(
    "This laptop has amazing performance but terrible battery life!", {"sentiment": ["positive", "negative", "neutral"]}
)
print(result)
# Output: {'sentiment': 'negative'}

# Multi-aspect classification
result = extractor.classify_text(
    "Great camera quality, decent performance, but poor battery life.",
    {
        "aspects": {
            "labels": ["camera", "performance", "battery", "display", "price"],
            "multi_label": True,
            "cls_threshold": 0.4,
        }
    },
)
print(result)
# Output: {'aspects': ['camera', 'performance', 'battery']}

{'sentiment': 'negative'}
{'aspects': ['camera', 'performance', 'battery']}


In [6]:
# Product information extraction
text = "iPhone 15 Pro Max with 256GB storage, A17 Pro chip, priced at $1199. Available in titanium and black colors."

result = extractor.extract_json(
    text,
    {
        "product": [
            "name::str::Full product name and model",
            "storage::str::Storage capacity like 256GB or 1TB",
            "processor::str::Chip or processor information",
            "price::str::Product price with currency",
            "colors::list::Available color options",
        ]
    },
)
print(result)
# Output: {
#     'product': [{
#         'name': 'iPhone 15 Pro Max',
#         'storage': '256GB',
#         'processor': 'A17 Pro chip',
#         'price': '$1199',
#         'colors': ['titanium', 'black']
#     }]
# }

# Multiple structured entities
text = "Apple Inc. headquarters in Cupertino launched iPhone 15 for $999 and MacBook Air for $1299."

result = extractor.extract_json(
    text,
    {
        "company": ["name::str::Company name", "location::str::Company headquarters or office location"],
        "products": ["name::str::Product name and model", "price::str::Product retail price"],
    },
)
print(result)
# Output: {
#     'company': [{'name': 'Apple Inc.', 'location': 'Cupertino'}],
#     'products': [
#         {'name': 'iPhone 15', 'price': '$999'},
#         {'name': 'MacBook Air', 'price': '$1299'}
#     ]
# }

{'product': [{'name': 'iPhone 15 Pro Max', 'storage': '256GB', 'processor': 'A17 Pro', 'price': '$1199', 'colors': ['black', 'titanium']}]}
{'company': [{'name': 'Apple Inc.', 'location': 'Cupertino'}], 'products': [{'name': 'iPhone 15', 'price': '$999'}, {'name': 'MacBook Air', 'price': '$1299'}]}


In [9]:
# 2️⃣ 테스트 쿼리
queries = [
    "이병헌이 나온 영화 추천해줘",
    "박찬욱 감독 작품 보여줘",
    "송강호랑 전도연이 같이 출연한 영화 있어?",
    "봉준호가 연출한 영화 찾아줘",
]

# 3️⃣ 스키마 정의
schema = {"cast_info": ["cast::list::배우, 출연자, 등장 인물의 이름", "director::list::감독, 연출자 이름"]}

# 4️⃣ 구조화된 정보 추출
for q in queries:
    result = extractor.extract_json(q, schema)
    print(f"🎬 Query: {q}")
    print(result)
    print("-" * 50)

🎬 Query: 이병헌이 나온 영화 추천해줘
{'cast_info': {}}
--------------------------------------------------
🎬 Query: 박찬욱 감독 작품 보여줘
{'cast_info': {}}
--------------------------------------------------
🎬 Query: 송강호랑 전도연이 같이 출연한 영화 있어?
{'cast_info': []}
--------------------------------------------------
🎬 Query: 봉준호가 연출한 영화 찾아줘
{'cast_info': {}}
--------------------------------------------------


In [11]:
# 2️⃣ 테스트 문장 (한국어)
texts = [
    "이병헌이 나온 영화 추천해줘",
    "박찬욱 감독 작품 보여줘",
    "봉준호와 송강호가 함께한 영화 알려줘",
    "전도연이 출연한 영화 있어?",
]

# 3️⃣ 영어 기반 스키마 (명확한 설명 추가)
labels = {
    "person": (
        "Names of people mentioned in text, including actors, directors, producers, "
        "or fictional characters appearing in movies or shows."
    )
}

# 4️⃣ 엔티티 추출
for text in texts:
    result = extractor.extract_entities(
        text,
        labels,
        threshold=0.3,  # threshold 낮추면 더 민감하게 인식
    )
    print(f"🗣️ Text: {text}")
    print(result)
    print("-" * 50)

🗣️ Text: 이병헌이 나온 영화 추천해줘
{'entities': {'person': []}}
--------------------------------------------------
🗣️ Text: 박찬욱 감독 작품 보여줘
{'entities': {'person': ['박찬욱']}}
--------------------------------------------------
🗣️ Text: 봉준호와 송강호가 함께한 영화 알려줘
{'entities': {'person': []}}
--------------------------------------------------
🗣️ Text: 전도연이 출연한 영화 있어?
{'entities': {'person': []}}
--------------------------------------------------


# Glinear-ko

https://huggingface.co/taeminlee/gliner_ko?utm_source=chatgpt.com

이름 추출 엄청 잘되네!

In [1]:
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [5]:
from gliner import GLiNER

model = GLiNER.from_pretrained("taeminlee/gliner_ko")

In [8]:
# 2️⃣ 영화 관련 예문들
texts = [
    "이병헌이 출연한 영화 추천해줘",
    "박찬욱 감독 작품 보여줘",
    "봉준호와 송강호가 함께한 영화 알려줘",
    "전도연이 나온 영화 중에 가장 유명한 건 뭐야?",
    "기생충은 어떤 감독이 만들었어?",
    "헤어질 결심의 주연 배우는 누구야?",
    "타이타닉에서 레오나르도 디카프리오가 맡은 역할은 뭐야?",
]

# 3️⃣ 한국어 전용 GLiNER-ko 라벨셋
labels = [
    "ARTIFACTS",  # 영화, 작품, 도서 등 (예: '기생충', '헤어질 결심')
    "PERSON",  # 인물 이름 (예: '이병헌', '박찬욱')
    "ORGANIZATION",  # 회사, 단체 (예: 'CJ ENM')
    "LOCATION",  # 장소 (예: '한국', '할리우드')
    "DATE",  # 연도나 날짜 (예: '2005년', '2023년')
]

# 4️⃣ 예문별 엔티티 추출
for text in texts:
    entities = model.predict_entities(text, labels)
    print(f"\n🎬 문장: {text}")
    for e in entities:
        print(f"  → {e['text']}  ⇒  {e['label']}")


🎬 문장: 이병헌이 출연한 영화 추천해줘
  → 이병헌  ⇒  PERSON
  → 추천해줘  ⇒  ARTIFACTS

🎬 문장: 박찬욱 감독 작품 보여줘
  → 박찬욱  ⇒  PERSON
  → 보여줘  ⇒  ARTIFACTS

🎬 문장: 봉준호와 송강호가 함께한 영화 알려줘
  → 봉준호  ⇒  PERSON
  → 송강호  ⇒  PERSON
  → 알려줘  ⇒  ARTIFACTS

🎬 문장: 전도연이 나온 영화 중에 가장 유명한 건 뭐야?
  → 전도연  ⇒  PERSON
  → 뭐야  ⇒  ARTIFACTS

🎬 문장: 기생충은 어떤 감독이 만들었어?
  → 기생충  ⇒  ARTIFACTS

🎬 문장: 헤어질 결심의 주연 배우는 누구야?
  → 헤어질 결심  ⇒  ARTIFACTS

🎬 문장: 타이타닉에서 레오나르도 디카프리오가 맡은 역할은 뭐야?
  → 타이타닉  ⇒  ARTIFACTS
  → 레오나르도 디카프리오  ⇒  PERSON


In [7]:
entities

[{'start': 1,
  'end': 8,
  'text': '피터 잭슨 경',
  'label': 'PERSON',
  'score': 0.9801418781280518},
 {'start': 11,
  'end': 26,
  'text': '1961년 10월 31일 ~',
  'label': 'DATE',
  'score': 0.9799976944923401},
 {'start': 30,
  'end': 34,
  'text': '뉴질랜드',
  'label': 'LOCATION',
  'score': 0.9998947381973267},
 {'start': 36,
  'end': 41,
  'text': '영화 감독',
  'label': 'CIVILIZATION',
  'score': 0.9779493808746338},
 {'start': 43,
  'end': 46,
  'text': '각본가',
  'label': 'CIVILIZATION',
  'score': 0.9986706972122192},
 {'start': 48,
  'end': 50,
  'text': '영화',
  'label': 'CIVILIZATION',
  'score': 0.7349614500999451},
 {'start': 51,
  'end': 55,
  'text': '프로듀서',
  'label': 'CIVILIZATION',
  'score': 0.943554699420929},
 {'start': 59,
  'end': 70,
  'text': 'J. R. R. 톨킨',
  'label': 'PERSON',
  'score': 0.9998692274093628},
 {'start': 84,
  'end': 97,
  'text': '반지의 제왕 영화 3부작',
  'label': 'ARTIFACTS',
  'score': 0.7942107319831848},
 {'start': 99,
  'end': 110,
  'text': '2001년~2003년',
  '